# GTAN – Semi-supervised Credit Card Fraud Detection
**Paper:** Semi-supervised Credit Card Fraud Detection via Attribute-Driven Graph Representation (AAAI 2023)

**Chạy trên Google Colab** — Runtime → Change runtime type → **GPU (T4)**

## 1. Cài dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cài PyTorch Geometric
import subprocess, sys

def run(cmd): subprocess.run(cmd, shell=True, check=True)

TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
PYG_URL = f'https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html'
print(f'Installing PyG for torch={TORCH} cuda={CUDA}')
print(f'Wheel URL: {PYG_URL}')

run('pip install torch_geometric -q')
run(f'pip install torch-scatter torch-sparse -f {PYG_URL} -q')
run('pip install scikit-learn pandas numpy scipy tqdm -q')
print('✓ Done')

## 2. Upload hoặc clone code

In [ ]:
# Option A: Upload file gtan_fraud.zip lên Colab rồi chạy cell này
# from google.colab import files
# uploaded = files.upload()   # chọn gtan_fraud.zip
# !unzip -q gtan_fraud.zip
# %cd gtan_fraud

# Option B: Clone từ GitHub nếu bạn đã push lên
# !git clone https://github.com/YOUR_USERNAME/gtan_fraud.git
# %cd gtan_fraud

# Sau khi cd vào folder, kiểm tra:
import os
print('Files:', os.listdir('.'))

## 3. Import modules

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_data
from trainer import gtan_train
print('✓ Import OK')

## 4. Config

In [ ]:
import torch

DATASET = 'yelp'   # 'yelp' | 'amazon' | 'sffsd'
DEVICE  = 'cuda:0' if torch.cuda.is_available() else 'cpu'

args = {
    'dataset'       : DATASET,
    'batch_size'    : 256,
    'hid_dim'       : 256,
    'lr'            : 0.003,
    'wd'            : 1e-4,
    'n_layers'      : 2,
    'dropout'       : [0.2, 0.1],
    'device'        : DEVICE,
    'early_stopping': 10,
    'n_fold'        : 5,      # giảm xuống 2 nếu muốn nhanh hơn
    'seed'          : 2023,
    'max_epochs'    : 100,    # giảm xuống 10 để test nhanh
    'gated'         : True,
    'test_size'     : 0.4,
    'checkpoint_dir': 'checkpoints',
}

print(f'Dataset: {DATASET}  |  Device: {DEVICE}')

## 5. Load data (tự động download)

In [ ]:
feat_df, labels, train_idx, test_idx, edge_index, cat_features = load_data(
    DATASET, data_dir='data', test_size=args['test_size']
)

print(f'Nodes        : {len(feat_df):,}')
print(f'Edges        : {edge_index.shape[1]:,}')
print(f'Features     : {feat_df.shape[1]}')
print(f'Train        : {len(train_idx):,}')
print(f'Test         : {len(test_idx):,}')
print(f'Fraud ratio  : {(labels==1).sum()/len(labels):.2%}')
print(f'Cat features : {cat_features}')

## 6. Train

In [ ]:
results = gtan_train(
    feat_df, edge_index, train_idx, test_idx,
    labels, args, cat_features
)

## 7. Kết quả

In [ ]:
print('='*40)
print(f'AUC      = {results["auc"]:.4f}  (paper YelpChi: 0.9241)')
print(f'F1-macro = {results["f1_macro"]:.4f}  (paper YelpChi: 0.7988)')
print(f'AP       = {results["ap"]:.4f}  (paper YelpChi: 0.7513)')
print('='*40)

## 8. Inference trên node mới

In [ ]:
import torch
import numpy as np

best_model = results['best_models'][0].to('cpu')
best_model.eval()

# Lấy vài node test để xem fraud probability
sample_idx = test_idx[:10]
x_sample   = torch.tensor(feat_df.iloc[sample_idx].values, dtype=torch.float32)
lbl_sample  = torch.full((len(sample_idx),), 2, dtype=torch.long)  # mask hết

# Edge index chỉ cho subgraph của sample nodes (đơn giản: dùng self-loop)
n = len(sample_idx)
sl = torch.arange(n)
ei = torch.stack([sl, sl])

with torch.no_grad():
    logits = best_model(x_sample, ei, lbl_sample, None)
    proba  = torch.softmax(logits, dim=1)[:, 1].numpy()

true_labels = labels.iloc[sample_idx].values
print('Node  | True | Fraud Prob')
print('-'*30)
for i, (t, p) in enumerate(zip(true_labels, proba)):
    flag = '← FRAUD' if t == 1 else ''
    print(f'  {i:3d} |  {t}   | {p:.4f}  {flag}')